In [1]:
import random
import json

#### JSON from the editor
```json
{
  "nodes": [
    {
      "width": 240,
      "height": 276,
      "id": "0.05272377425322505-1756500835216",
      "type": "eventType",
      "position": {
        "x": 145.53354881065553,
        "y": 1196.7836919286149
      },
      "data": {
        "box": {
          "newEventVars": {
            "0": [
              "payment reminder"
            ],
            "1": [
              "pay order"
            ]
          },
          "newObjectVars": {
            "0": [
              "orders"
            ]
          },
          "filters": [
            {
              "type": "O2E",
              "object": 0,
              "event": 0,
              "qualifier": null
            },
            {
              "type": "O2E",
              "object": 0,
              "event": 1,
              "qualifier": null
            }
          ],
          "sizeFilters": [],
          "constraints": [
            {
              "type": "Filter",
              "filter": {
                "type": "TimeBetweenEvents",
                "from_event": 0,
                "to_event": 1,
                "min_seconds": 0,
                "max_seconds": null
              }
            }
          ],
          "evVarLabels": {},
          "obVarLabels": {}
        }
      },
      "positionAbsolute": {
        "x": 145.53354881065553,
        "y": 1196.7836919286149
      },
      "selected": true,
      "dragging": false
    }
  ],
  "edges": []
}
```

#### JSON given to the DB
```json
{
  "nodes": [
    {
      "Box": [
        {
          "newEventVars": { "0": ["payment reminder"], "1": ["pay order"] },
          "newObjectVars": { "0": ["orders"] },
          "filters": [
            { "type": "O2E", "object": 0, "event": 0, "qualifier": null },
            { "type": "O2E", "object": 0, "event": 1, "qualifier": null }
          ],
          "sizeFilters": [],
          "constraints": [],
          "evVarLabels": {},
          "obVarLabels": {}
        },
        []
      ]
    }
  ],
  "edgeNames": []
}
```

In [ ]:
patterns = [
    "EVENT_A happens before EVENT_B",
    "Whenever EVENT_A occurs, EVENT_B eventually occurs",
    "After EVENT_A, EVENT_B must not occur",
    "EVENT_A and EVENT_B refer to the same OBJECT",
    "EVENT_B happens within TIME hours after EVENT_A"
]

domains = {
    # Generic domains
    "e-commerce": {
        "events": ["create order", "add item", "pay order", "ship package", "cancel order"],
        "objects": ["customers", "employees", "items", "orders", "packages", "products"]
    },
    "recruitment": {
        "events": ["submit application", "review application", "approve application", "reject application"],
        "objects": ["applications", "candidates"]
    },
    "software_dev": {
        "events": ["open issue", "submit pull request", "review pull request", "merge pull request", "close issue"],
        "objects": ["issues", "pull requests", "commits"]
    },

    # Domains from actual log files
    "distribution": {
        "events": ["Apply Discount", "Apply Payment", "Approve Order", "Approve Return", "Generate Invoice", 
                   "Generate Pick List", "Generate Shipping Label", "Handle Payment Discrepancy", 
                   "Issue Credit Note", "Issue Payment Reminder", "Pack Items", "Perform Credit Check", 
                   "Pick Items", "Process Return Request", "Receive Customer Order", "Receive Payment", 
                   "Reject Order", "Schedule Order Fulfillment", "Send Invoice to Customer", "Ship Order", 
                   "Update Inventory", "Validate Customer Order"],
        "objects": ["CreditNote", "Customer", "Dispute", "Invoice", "Order", "Payment", "Product", "Return", "Shipment"]
    },
    "procurement": {
        "events": ["Approve PO", "Approve PR", "Create PO", "Create PO (Batch)", "Create PR", "Make Payment", 
                   "Receive Goods", "Receive Invoice", "Reject PR", "Send PO to Supplier", 
                   "Three-way Match - Mismatch Found", "Three-way Match - Successful"],
        "objects": ["Goods Receipt", "Purchase Requisition", "Payment", "Invoice", "Purchase Order", "Supplier"]
    },
    "02_p2p": {
        "events": ["Approve PO", "Approve PR", "Create PO", "Create PO (Batch)", "Create PR", "Make Payment", 
                   "Receive Goods", "Receive Invoice", "Reject PR", "Send PO to Supplier", "Three-way Match - Mismatch Found", 
                   "Three-way Match - Successful"],
        "objects": ["Goods Receipt", "Invoice", "Payment", "Purchase Order", "Purchase Requisition", "Supplier"]
    },
    "hiring": {
        "events": ['Approve Job Requisition', 'Attend Orientation Training', 'Candidate Accept Offer', 'Candidate Reject Offer', 
                   'Complete Assessment', 'Complete Background Check', 'Complete HR Paperwork', 'Complete Onboarding', 
                   'Conduct Initial Interview', 'Conduct Panel Interview', 'Create Job Requisition', 'Evaluate Interview', 
                   'Initiate Background Check', 'Post Job Opening', 'Prepare Offer Letter', 'Review Assessment', 
                   'Schedule Initial Interview', 'Schedule Panel Interview', 'Screen Resume', 'Send Assessment', 
                   'Send Offer Letter', 'Send Rejection Letter', 'Start Onboarding', 'Submit Application'],
        "objects": ['Interviewer', 'HiringManager', 'Recruiter', 'Assessment', 'Candidate', 'JobRequisition']
    },
    "hospital": {
        "events": ['Appointment Scheduling', 'Assign Bed', 'Diagnostic Test Order', 'Initial Evaluation', 
                   'Insurance Verification', 'Patient Arrival', 'Perform Diagnostic Test', 'Pre-Admission Consultation', 
                   'Registration'],
        "objects": ['Patient', 'InsuranceCompany', 'Physician', 'LabTechnician', 'Nurse', 'DiagnosticEquipment', 'Room', 'Bed']
    },
    "container_logistics": {
        "events": ['Book Vehicles', 'Bring to Loading Bay', 'Collect Goods', 'Create Transport Document', 'Depart', 
                   'Drive to Terminal', 'Load Truck', 'Load to Vehicle', 'Order Empty Containers', 'Pick Up Empty Container', 
                   'Place in Stock', 'Register Customer Order', 'Reschedule Container', 'Weigh'],
        "objects": ['Container', 'Customer Order', 'Forklift', 'Handling Unit', 'Transport Document', 'Truck', 'Vehicle']
    },
    "lending": {
        "events": ['Application Rejected', 'Appointment Denied', 'Hand In Credit Appliaction', 'Loan Denied', 
                   'Make Visit to Assess Colatteral', 'Request Appointment', 'Request Co-Signer On Loan', 
                   'Set Appointment', 'Sign Loan Agreement', 'Submit File to Underwriter', 'Verify Borrowers Information', 
                   'skipped_examination'],
        "objects": ['Person']
    },
    "renting": {
        "events": ["Accept Late Payment", "Apply for Viewing Appointment", "Check Paperwork", "Evict Tenant", "Extensive Screening", 
                   "Hand In The Paperwork", "Issue Warning", "Miss Rent Payment", "Move In", "Pay Rent", "Reject Prospective Tenant", 
                   "Screen Prospective Tenant", "Set Appointment", "Sign Contract", "Tenant Cancels Appartment", "View The Property"],
        "objects": ["Person"]
    },
    "container_logistics": {
        "events": ['Book Vehicles', 'Bring to Loading Bay', 'Collect Goods', 'Create Transport Document', 'Depart', 
                   'Drive to Terminal', 'Load Truck', 'Load to Vehicle', 'Order Empty Containers', 'Pick Up Empty Container', 
                   'Place in Stock', 'Register Customer Order', 'Reschedule Container', 'Weigh'],
        "objects": ['Container', 'Customer Order', 'Forklift', 'Handling Unit', 'Transport Document', 'Truck', 'Vehicle']
    },
}

In [3]:
def generate_example():
    pattern = random.choice(patterns)
    
    # Pick a domain first to ensure semantic consistency
    domain = random.choice(list(domains.keys()))
    domain_data = domains[domain]

    A = random.choice(domain_data["events"])
    # Ensure event A and B are different for better logical examples
    B = random.choice([e for e in domain_data["events"] if e != A])
    obj = random.choice(domain_data["objects"])
    t = random.choice([1, 6, 24, 72])

    text = pattern.replace("EVENT_A", A)\
                  .replace("EVENT_B", B)\
                  .replace("OBJECT", obj)\
                  .replace("TIME", str(t))

    # Dynamically inject constraints based on the pattern
    constraints = []
    filters = []
    if "before" in pattern or "eventually" in pattern:
        constraints = [{
            "type": "Filter",
            "filter": { "type": "TimeBetweenEvents", "from_event": 0, "to_event": 1, "min_seconds": 0, "max_seconds": None }
        }]
    elif "within TIME hours" in pattern:
        constraints = [{
            "type": "Filter",
            "filter": { "type": "TimeBetweenEvents", "from_event": 0, "to_event": 1, "min_seconds": 0, "max_seconds": t * 3600 }
        }]
    elif "must not occur" in pattern:
        # Requires a child edge with a NOT constraint to be completely valid OCPQ
        constraints = [{
            "type": "NOT",
            "child_names": ["edge_1"]
        }]
    elif "refer to the same OBJECT" in pattern:
        filters = [
            { "type": "O2E", "object": 0, "event": 0, "qualifier": None },
            { "type": "O2E", "object": 0, "event": 1, "qualifier": None }
        ]

    ocpq_tree = {
            "nodes": [
                {
                    "Box": [
                        {
                            "newEventVars": { "0": [A], "1": [B] },
                            "newObjectVars": { "0": [obj] },
                            "filters": filters,
                            "sizeFilters": [],
                            "constraints": constraints,
                            "evVarLabels": {},
                            "obVarLabels": {}
                        },
                        [] # IDs of any connected child nodes in the tree
                    ]
                }
            ],
            "edgeNames": []
    }

    return {"nl": text, "query": ocpq_tree}

dataset = []
seen = set()

while len(dataset) < 200:
    example = generate_example()
    # Use the natural language text to check for uniqueness
    if example["nl"] not in seen:
        seen.add(example["nl"])
        dataset.append(example)

with open("ocpq_dataset.json","w") as f:
    json.dump(dataset,f,indent=2)

In [4]:
dataset[:5]

[{'nl': 'review application and submit application refer to the same candidates',
  'query': {'nodes': [{'Box': [{'newEventVars': {'0': ['review application'],
        '1': ['submit application']},
       'newObjectVars': {'0': ['candidates']},
       'filters': [{'type': 'O2E', 'object': 0, 'event': 0, 'qualifier': None},
        {'type': 'O2E', 'object': 0, 'event': 1, 'qualifier': None}],
       'sizeFilters': [],
       'constraints': [],
       'evVarLabels': {},
       'obVarLabels': {}},
      []]}],
   'edgeNames': []}},
 {'nl': 'After ship package, add item must not occur',
  'query': {'nodes': [{'Box': [{'newEventVars': {'0': ['ship package'],
        '1': ['add item']},
       'newObjectVars': {'0': ['items']},
       'filters': [],
       'sizeFilters': [],
       'constraints': [{'type': 'NOT', 'child_names': ['edge_1']}],
       'evVarLabels': {},
       'obVarLabels': {}},
      []]}],
   'edgeNames': []}},
 {'nl': 'create order and cancel order refer to the same items',